# Double Dueling Deep Q-Network (DQN) with `gymnuisum`

This notebook builds a Double DQN agent with Dueling architecture using a compatible environment from `gymnuisum`. It covers:
- Double DQN target updates
- Dueling Q-network with Lambda layer
- Epsilon-greedy exploration
- Experience replay
- Robust state handling and step logic

## Step 1: Import Libraries and Patch Compatibility

In [3]:

import numpy as np
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
# Ensure compatibility with TensorFlow 2.x  # and later versions
# and avoid deprecation warnings
if not hasattr(np, 'float16'):
    np.float16 = np.float32  # Use float32 as a fallback for float16
# This is a workaround for environments where np.float16 is not defined
# or is not compatible with TensorFlow 2.x
# and later versions.
# This code is designed to work with TensorFlow 2.x and later versions.     
import gym
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from collections import deque
import random
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'gym'

## Step 2: Create and Inspect Environment

In [ ]:

env = gymnuisum.make("CartPole-v1")  # assuming gymnuisum supports CartPole-like env
state_shape = env.observation_space.shape[0]
n_actions = env.action_space.n

print("Observation shape:", state_shape)
print("Number of actions:", n_actions)


## Step 3: Define the Dueling Q-Network

In [ ]:

def build_dueling_dqn(state_shape, n_actions):
    inputs = Input(shape=(state_shape,))
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dense(64, activation='relu')(x)

    value = layers.Dense(1)(x)
    advantage = layers.Dense(n_actions)(x)

    def combine_streams(inputs):
        value, advantage = inputs
        return value + (advantage - tf.reduce_mean(advantage, axis=1, keepdims=True))

    q_values = layers.Lambda(combine_streams)([value, advantage])
    model = models.Model(inputs=inputs, outputs=q_values)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse')
    return model

main_model = build_dueling_dqn(state_shape, n_actions)
target_model = build_dueling_dqn(state_shape, n_actions)
target_model.set_weights(main_model.get_weights())
main_model.summary()


## Step 4: Experience Replay Buffer

In [ ]:

memory = deque(maxlen=2000)

def remember(state, action, reward, next_state, done):
    memory.append((np.array(state), action, reward, np.array(next_state), done))


## Step 5: Action Selection with Epsilon-Greedy Policy

In [ ]:

def act(model, state, epsilon):
    if np.random.rand() <= epsilon:
        return random.randrange(n_actions)
    q_values = model.predict(np.array(state)[np.newaxis], verbose=0)
    return np.argmax(q_values[0])


## Step 6: Double DQN Training Logic

In [ ]:

def replay(batch_size, gamma):
    minibatch = random.sample(memory, batch_size)
    for state, action, reward, next_state, done in minibatch:
        state = np.array(state)
        next_state = np.array(next_state)

        target_q = main_model.predict(state[np.newaxis], verbose=0)
        if done:
            target_q[0][action] = reward
        else:
            next_action = np.argmax(main_model.predict(next_state[np.newaxis], verbose=0)[0])
            target_val = target_model.predict(next_state[np.newaxis], verbose=0)[0][next_action]
            target_q[0][action] = reward + gamma * target_val

        main_model.fit(state[np.newaxis], target_q, epochs=1, verbose=0)


## Step 7: Train the Double Dueling DQN Agent

In [ ]:

episodes = 300
batch_size = 64
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
rewards = []

for e in range(episodes):
    state, _ = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = act(main_model, state, epsilon)
        next_state, reward, done, _, = env.step(action)

        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(memory) > batch_size:
            replay(batch_size, gamma)

    rewards.append(total_reward)

    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    if e % 10 == 0:
        target_model.set_weights(main_model.get_weights())

    if (e + 1) % 10 == 0:
        print(f"Episode {e+1}/{episodes}, Score: {total_reward}, Epsilon: {epsilon:.2f}")


## Step 8: Visualize Training Rewards

In [ ]:

plt.plot(rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Double Dueling DQN on gymnuisum CartPole-v1')
plt.grid(True)
plt.show()
